In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=4, suppress=True)


## Task 01 — Statistics along axes, on multi-dimensional data

In [2]:
# 4 shops, 30 days, 3 product categories
rng = np.random.default_rng(42)
sales = rng.integers(50, 500, size=(4, 30, 3))
print("sales shape:", sales.shape)

# --- Total sales per shop: collapse days (axis=1) AND categories (axis=2) ---
per_shop = sales.sum(axis=(1, 2))
print("per_shop shape:", per_shop.shape)   # (4,)
print("per_shop:", per_shop)

# --- Average daily sales per category, across all shops and days ---
# collapse shop (axis=0) and day (axis=1), keep category (axis=2)
avg_per_category = sales.mean(axis=(0, 1))
print("avg_per_category shape:", avg_per_category.shape)  # (3,)
print("avg_per_category:", avg_per_category)

# --- Best day per shop: collapse categories first, then argmax over days ---
daily_totals = sales.sum(axis=2)          # shape (4, 30) -> total per shop per day
print("daily_totals shape:", daily_totals.shape)
best_day_per_shop = daily_totals.argmax(axis=1)   # shape (4,)
print("best_day_per_shop shape:", best_day_per_shop.shape)
print("best_day_per_shop (0-indexed day numbers):", best_day_per_shop)

# --- Most consistent shop: lowest std of daily totals ---
std_per_shop = daily_totals.std(axis=1)   # shape (4,)
print("std_per_shop shape:", std_per_shop.shape)
print("std_per_shop:", std_per_shop)
most_consistent_shop = std_per_shop.argmin()
print("most consistent shop index:", most_consistent_shop)

# --- keepdims=True demo: mean-centre the original array ---
# mean across days (axis=1), per shop per category, keeping the dim so it broadcasts
day_mean_keepdims = sales.mean(axis=1, keepdims=True)
print("day_mean_keepdims shape:", day_mean_keepdims.shape)   # (4, 1, 3)

centred = sales - day_mean_keepdims       # broadcasts cleanly because of keepdims
print("centred shape:", centred.shape)    # (4, 30, 3)
print("centred mean along days (~0):", centred.mean(axis=1))


sales shape: (4, 30, 3)
per_shop shape: (4,)
per_shop: [25751 24730 24062 24356]
avg_per_category shape: (3,)
avg_per_category: [274.625  277.7667 271.7667]
daily_totals shape: (4, 30)
best_day_per_shop shape: (4,)
best_day_per_shop (0-indexed day numbers): [15 23  7 25]
std_per_shop shape: (4,)
std_per_shop: [177.5354 205.113  191.2724 223.6728]
most consistent shop index: 0
day_mean_keepdims shape: (4, 1, 3)
centred shape: (4, 30, 3)
centred mean along days (~0): [[ 0. -0. -0.]
 [-0.  0.  0.]
 [-0.  0. -0.]
 [ 0. -0.  0.]]


**Look it up — answers**

- `axis=1` on a `(4, 30, 3)` array collapses the **days** axis, leaving shape `(4, 3)` (shop, category).
- Without `keepdims=True`, the reduced axis is dropped entirely, so subtracting a `(4, 3)` mean from a
  `(4, 30, 3)` array fails to broadcast (shapes don't align) — you'd get a `ValueError`. `keepdims=True`
  keeps that axis as size 1 (`(4, 1, 3)`), which broadcasts cleanly against `(4, 30, 3)`.
- Yes — `axis` accepts a tuple. `axis=(1, 2)` reduces over both axis 1 and axis 2 simultaneously
  (days *and* categories here), leaving only the shop axis.

## Task 02 — Solve a linear system with Ax = b

In [ ]:
A = np.array([[2., 3., -1.],
              [4., -1., 2.],
              [-1., 2., 3.]])
b = np.array([5., 6., 7.])

x = np.linalg.solve(A, b)
print("x =", x)
print("A @ x == b (allclose)?", np.allclose(A @ x, b))



x_inv = np.linalg.inv(A) @ b
print("x_inv =", x_inv)
print("solve vs inv allclose?", np.allclose(x, x_inv))

# --- Time both approaches on a 500x500 random system ---
rng = np.random.default_rng(0)
A_big = rng.random((500, 500))
b_big = rng.random(500)

t0 = time.perf_counter()
x_solve = np.linalg.solve(A_big, b_big)
t_solve = time.perf_counter() - t0

t0 = time.perf_counter()
x_inv_big = np.linalg.inv(A_big) @ b_big
t_inv = time.perf_counter() - t0

print(f"solve: {t_solve:.5f}s  |  inv@b: {t_inv:.5f}s  |  ratio (inv/solve): {t_inv / t_solve:.2f}x")

# --- Singular matrix ---
A_singular = np.array([[1., 2., 3.],
                        [2., 4., 6.],   # row 2 = 2 * row 1
                        [1., 0., 1.]])
try:
    np.linalg.solve(A_singular, np.array([1., 2., 3.]))
except np.linalg.LinAlgError as e:
    print("LinAlgError:", e)
    # Meaning: determinant is 0 -> matrix has no inverse -> the system either has
    # no solution or infinitely many; solve() can't return a single unique answer.


x = [1.0476 1.5238 1.6667]
A @ x == b (allclose)? True
x_inv = [1.0476 1.5238 1.6667]
solve vs inv allclose? True
solve: 0.01824s  |  inv@b: 0.03578s  |  ratio (inv/solve): 1.96x
LinAlgError: Singular matrix


**Look it up — answers**

- `solve` is preferred over `inv`: (1) it's numerically more stable (fewer rounding errors, avoids
  explicitly forming the ill-conditioned inverse), and (2) it's computationally cheaper — `solve` is
  roughly 2-3x faster since it does LU decomposition once, while `inv` does extra work to build the
  full inverse matrix before you even multiply by `b`.
- A determinant of zero means the matrix is **singular** — it has no inverse, and the system `Ax = b`
  either has **no solution** or **infinitely many solutions** (the rows/columns are linearly dependent).
- In ML, a singular (or near-singular) feature matrix usually means **multicollinearity** — some
  features are exact or near-exact linear combinations of others, which breaks closed-form solutions
  like the normal equation and makes coefficients unstable.

## Task 03 — Cosine similarity from scratch

In [4]:
def cosine_similarity(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

identical = np.array([1., 2., 3.])
opposite = -identical
perpendicular_a = np.array([1., 0.])
perpendicular_b = np.array([0., 1.])

print("identical vs identical:", cosine_similarity(identical, identical))
print("identical vs opposite:", cosine_similarity(identical, opposite))
print("perpendicular pair:", cosine_similarity(perpendicular_a, perpendicular_b))

assert np.isclose(cosine_similarity(identical, identical), 1.0)
assert np.isclose(cosine_similarity(identical, opposite), -1.0)
assert np.isclose(cosine_similarity(perpendicular_a, perpendicular_b), 0.0)
print("all three cases verified with np.isclose")

# --- Vectorised: similarity of 100 vectors against ONE query vector, no loop ---
rng = np.random.default_rng(1)
M = rng.normal(size=(100, 16))       # 100 vectors, 16-dim
query = rng.normal(size=16)

sims_to_query = (M @ query) / (np.linalg.norm(M, axis=1) * np.linalg.norm(query))
print("sims_to_query shape:", sims_to_query.shape)   # (100,)

# --- Full 100x100 pairwise similarity matrix, no loops ---
norms = np.linalg.norm(M, axis=1, keepdims=True)   # (100, 1) -- keepdims to broadcast!
M_normalised = M / norms
similarity_matrix = M_normalised @ M_normalised.T   # (100, 100)
print("similarity_matrix shape:", similarity_matrix.shape)
print("diagonal (self-similarity, should all be ~1):", np.allclose(np.diag(similarity_matrix), 1.0))

# Why cosine similarity over Euclidean distance for text embeddings:
# Embedding magnitude often just reflects text length / word frequency, not meaning.
# Cosine similarity ignores magnitude and compares only direction, so it captures
# semantic similarity regardless of how long the texts are.


identical vs identical: 1.0
identical vs opposite: -1.0
perpendicular pair: 0.0
all three cases verified with np.isclose
sims_to_query shape: (100,)
similarity_matrix shape: (100, 100)
diagonal (self-similarity, should all be ~1): True


## Task 04 — Generate synthetic 2D data with controllable noise

In [5]:
def make_line(n, slope, intercept, noise, seed=0):
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 10, n)
    y = slope * x + intercept + rng.normal(0, noise, n)
    return x, y

def make_clusters(n, centres, spread, seed=0):
    """centres: array-like of shape (k, 2). Returns points (n*k, 2) and labels (n*k,)."""
    rng = np.random.default_rng(seed)
    centres = np.asarray(centres)
    k = centres.shape[0]
    points = np.concatenate([
        centres[i] + rng.normal(0, spread, size=(n, 2)) for i in range(k)
    ])  # small python loop over CLUSTERS (k is tiny, e.g. 2) -- the per-point work is vectorised
    labels = np.repeat(np.arange(k), n)
    return points, labels

noise_levels = [0.5, 2.0, 10.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, noise in zip(axes, noise_levels):
    x, y = make_line(60, slope=3.0, intercept=1.0, noise=noise, seed=42)
    est_slope, est_intercept = np.polyfit(x, y, 1)
    ax.scatter(x, y, s=12)
    ax.plot(x, est_slope * x + est_intercept, color="red")
    ax.set_title(f"noise={noise}  |  recovered slope={est_slope:.2f}")
plt.tight_layout()
plt.show()
# At noise=10.0 (relative to a slope*x range of ~0-30), recovery gets visibly unreliable --
# the estimated slope drifts noticeably from the true value of 3.0.

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

overlap_pts, overlap_labels = make_clusters(150, centres=[[0, 0], [1.5, 1.5]], spread=1.5, seed=1)
axes[0].scatter(overlap_pts[:, 0], overlap_pts[:, 1], c=overlap_labels, cmap="coolwarm", s=12)
axes[0].set_title("overlapping clusters")

separate_pts, separate_labels = make_clusters(150, centres=[[0, 0], [10, 10]], spread=1.0, seed=1)
axes[1].scatter(separate_pts[:, 0], separate_pts[:, 1], c=separate_labels, cmap="coolwarm", s=12)
axes[1].set_title("clearly separated clusters")
plt.tight_layout()
plt.show()

# Seeded everything above (seed=42 for lines, seed=1 for clusters) -- re-running this cell
# reproduces identical plots every time.


In [ ]:
# 1) Float addition is not exact
print("0.1 + 0.2 =", repr(0.1 + 0.2))
print("1.1 - 0.2 - 0.9 =", repr(1.1 - 0.2 - 0.9))  
print("0.3 - 0.2 =", repr(0.3 - 0.2))              

# 2) Integer overflow
small = np.array([1, 2, 3], dtype=np.int8)  
print("small * 100 =", small * 100)         

# 3) float32 precision loss
x = np.float32(1e8)
print("float32(1e8 + 1) - 1e8 =", np.float32(x + 1) - x)   # should be 1.0, but isn't

# 4) Catastrophic cancellation
a32 = np.float32(100000.3)
b32 = np.float32(100000.2)
print("a32 - b32 =", a32 - b32, "  (true answer is 0.1)")

# 5) Summation order / dtype effect over many floats
million = np.full(1_000_000, 0.1, dtype=np.float32)
exact = 100_000.0   # 1,000,000 * 0.1

sum32 = million.sum()
sum64 = million.astype(np.float64).sum()

print(f"float32 sum: {sum32:.6f}  (error: {abs(sum32 - exact):.6f})")
print(f"float64 sum: {sum64:.6f}  (error: {abs(sum64 - exact):.6f})")
print("float64 is closer" if abs(sum64 - exact) < abs(sum32 - exact) else "float32 is closer")

print("\nmachine epsilon float32:", np.finfo(np.float32).eps)
print("machine epsilon float64:", np.finfo(np.float64).eps)



0.1 + 0.2 = 0.30000000000000004
1.1 - 0.2 - 0.9 = 1.1102230246251565e-16
0.3 - 0.2 = 0.09999999999999998
small * 100 = [100 -56  44]
float32(1e8 + 1) - 1e8 = 0.0
a32 - b32 = 0.09375   (true answer is 0.1)
float32 sum: 100000.007812  (error: 0.007812)
float64 sum: 100000.001490  (error: 0.001490)
float64 is closer

machine epsilon float32: 1.1920929e-07
machine epsilon float64: 2.220446049250313e-16


X shape: (300, 4)
standardised means ~0: True
standardised stds ~1: True
cov shape: (4, 4)
cov symmetric? True
eigenvalues (descending): [2.0999 1.0101 0.8469 0.0564]
variance ratio per component: [0.5232 0.2517 0.211  0.0141]
cumulative variance: [0.5232 0.7749 0.9859 1.    ]
components needed for 90% variance: 3
X_projected shape: (300, 2)
np.cov matches ours? True
eigenvalues match np.cov's? True


loops: 1.963s  |  @: 0.00177s  |  @ is 1106x faster
results match? True
normal equation [slope, intercept]: [ 2.5305 -1.2843]
np.polyfit [slope, intercept]: [ 2.5305 -1.2843]
condition number (identity): 1.0
condition number (ill): 90004.00008871953
solution shift from a 1e-6 perturbation in b: [0.02 0.01 0.01]
k= 1  relative reconstruction error: 0.4968
k= 5  relative reconstruction error: 0.4284
k=10  relative reconstruction error: 0.3513
k=25  relative reconstruction error: 0.1626
k=50  relative reconstruction error: 0.0000
